# Credit Card Customer Spending Analysis

**ACC102 Track 2: GitHub Data Analysis Project**

This notebook studies how income, age, and home ownership relate to average monthly credit card spending. The intended user is a junior credit card product analyst who needs a simple customer segmentation view for marketing and basic monitoring.

## 1. Problem Definition

**Analytical question:** Which customer characteristics are most strongly associated with higher average monthly credit card spending?

**Target user:** A junior credit card product analyst.

**Business value:** The output helps the analyst understand whether customer segmentation should focus more on income, age, or home ownership.

## 2. Data Source

The dataset is William Greene's credit card data, available through `statsmodels.datasets.ccard` and based on Greene's *Econometric Analysis*.

- Access date: 27 April 2026
- Observations: 72 customers
- Variables used: average monthly expenditure, age, income, and home ownership

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "credit_card_spending.csv"
OUTPUT_DIR = ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load and Inspect the Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()

## 4. Data Cleaning and Feature Preparation

The raw data is already clean, but the original column names are short. I rename them into clearer business labels and create two additional variables: income band and spend-to-income ratio.

In [ ]:
df.columns = df.columns.str.lower()
df = df.rename(
    columns={
        "avgexp": "avg_monthly_spend",
        "age": "age",
        "income": "income_thousand",
        "ownrent": "home_owner",
    }
)

df["home_owner_label"] = df["home_owner"].map({1: "Owner", 0: "Renter"})
df["income_band"] = pd.qcut(
    df["income_thousand"],
    q=3,
    labels=["Low income", "Middle income", "High income"],
)
df["spend_to_income_ratio"] = df["avg_monthly_spend"] / (df["income_thousand"] * 1000)

df.head()

## 5. Descriptive Analysis

In [ ]:
summary_statistics = df[["avg_monthly_spend", "age", "income_thousand", "spend_to_income_ratio"]].describe().round(2)
summary_statistics

In [ ]:
income_band_summary = (
    df.groupby("income_band", observed=True)
    .agg(
        customers=("avg_monthly_spend", "size"),
        avg_spend=("avg_monthly_spend", "mean"),
        median_spend=("avg_monthly_spend", "median"),
        avg_income_thousand=("income_thousand", "mean"),
        avg_spend_to_income_ratio=("spend_to_income_ratio", "mean"),
    )
    .round(3)
)
income_band_summary

In [ ]:
home_ownership_summary = (
    df.groupby("home_owner_label")
    .agg(
        customers=("avg_monthly_spend", "size"),
        avg_spend=("avg_monthly_spend", "mean"),
        median_spend=("avg_monthly_spend", "median"),
        avg_income_thousand=("income_thousand", "mean"),
    )
    .round(2)
)
home_ownership_summary

## 6. Visualisation

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column="avg_monthly_spend", by="income_band", ax=ax, grid=False)
ax.set_title("Credit Card Spending by Income Band")
ax.set_xlabel("Income band")
ax.set_ylabel("Average monthly spending")
fig.suptitle("")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "spending_by_income_band.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = df["home_owner"].map({1: "#2F6F73", 0: "#D9822B"})
ax.scatter(
    df["income_thousand"],
    df["avg_monthly_spend"],
    c=colors,
    alpha=0.82,
    edgecolor="white",
    linewidth=0.6,
)
ax.set_title("Income and Monthly Credit Card Spending")
ax.set_xlabel("Income (thousand)")
ax.set_ylabel("Average monthly spending")
ax.text(0.02, 0.95, "Teal = homeowner, orange = renter", transform=ax.transAxes, va="top", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "income_vs_spending.png", dpi=200)
plt.show()

In [ ]:
owner_summary_plot = home_ownership_summary["avg_spend"].reindex(["Renter", "Owner"])

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(owner_summary_plot.index, owner_summary_plot.values, color=["#D9822B", "#2F6F73"])
ax.set_title("Average Spending by Home Ownership")
ax.set_xlabel("Home ownership")
ax.set_ylabel("Average monthly spending")
for i, value in enumerate(owner_summary_plot.values):
    ax.text(i, value + 8, f"{value:.0f}", ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "spending_by_home_ownership.png", dpi=200)
plt.show()

## 7. Simple Regression Model

This model is used for interpretation, not for real credit approval. It estimates how monthly spending is associated with income, age, and home ownership.

In [ ]:
model = smf.ols("avg_monthly_spend ~ income_thousand + age + home_owner", data=df).fit()
model.summary()

In [ ]:
regression_results = pd.DataFrame({
    "coefficient": model.params,
    "p_value": model.pvalues,
}).round(4)
regression_results

## 8. Save Outputs

In [ ]:
summary_statistics.to_csv(OUTPUT_DIR / "summary_statistics.csv")
income_band_summary.to_csv(OUTPUT_DIR / "income_band_summary.csv")
home_ownership_summary.to_csv(OUTPUT_DIR / "home_ownership_summary.csv")
regression_results.to_csv(OUTPUT_DIR / "regression_results.csv")
with open(OUTPUT_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
    f.write(model.summary().as_text())

print("Outputs saved to", OUTPUT_DIR)

## 9. Interpretation

The analysis suggests that income is the most useful segmentation variable in this dataset. Average monthly spending increases across the low, middle, and high income groups. The regression model also shows a positive income coefficient. Age and home ownership appear less important after controlling for income.

For a credit card product analyst, the practical implication is that income-based segmentation may be more useful than age-based segmentation for designing spending campaigns. However, the dataset is small and educational, so the findings should be treated as exploratory rather than as a basis for real credit decisions.